# Adverse Events Data Pipeline

---
## Overview
Notebook 2 of 8.

This notebook ingests adverse event data into the Bronze layer, maintaining data lineage and auditability.

Run the entire notebook to load the data.

---

## Source Data

- **File:** `ae.csv`
- **Location:** A Public GitHub repository set up especially for this exercise
- **Note:** Uses HTTPS URL so the notebook runs across any Databricks workspace

---

## Processing Steps

**1. Environment Setup**  
Ensures the required catalog and schemas exist for the Bronze layer

**2. Data Ingestion**  
Loads adverse event records into a DataFrame  
*Note: Uses Pandas to read from GitHub HTTPS URL*

**3. Provenance Metadata**  
Adds tracking metadata:
- Ingestion timestamp
- Unique ingest ID  
- Source filename

---

## Target

Data written to **bronze.ae_raw** Delta table using Spark MERGE for incremental updates.

---

In [0]:
from pyspark.sql.functions import lit, current_timestamp
from delta.tables import DeltaTable
from datetime import datetime
import pandas as pd

In [0]:
#global variables
#do not change the github link - this is a public repo from where the source data can be reliably sourced into any databricks workspace
CATALOG = "cluepoints_db_demo"
SCHEMA = "bronze"
GITHUB_URL = "https://raw.githubusercontent.com/prolfdog/cp_db_demo/main/"
ingestion_id = datetime.now().strftime('%Y%m%d_%H%M%S')
loaded_at = current_timestamp()
table_name = "ae"
full_table_name = f"{CATALOG}.{SCHEMA}.{table_name}"

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

DataFrame[]

In [0]:
#use pandas to import from github as spark can't read https urls - NOT FOR PRODUCTION, ONLY FOR DEMO
#in production we would use spark.read.csv
#drop unnamed column on import
df_demog = spark.createDataFrame(
    pd.read_csv(f"{GITHUB_URL}ae.csv").drop(columns=['Unnamed: 0'])
)

In [0]:
#add data provenance variables to the data frame
df_demog = (df_demog 
    .withColumn("source_file", lit("ae.csv"))
    .withColumn("loaded_at", lit(loaded_at))
    .withColumn("ingestion_id", lit(ingestion_id))
)

In [0]:
if not spark.catalog.tableExists(full_table_name):
    #create the table if it doesnt exist
    df_demog.write.format("delta").saveAsTable(full_table_name)
else:
    #from analysing the data we know that a row is unique by subjid, hlgtn and aestrdatetime
    #but note also that aestrdatetime is MISSING for some rows
    #aestrdatetime is included in the merge condition even though its sometimes null - observed the correct row count in the target - 720 rows
    target = DeltaTable.forName(spark, full_table_name)
    target.alias("target").merge(df_demog.alias("source"),
    "target.subjid = source.subjid AND target.aestrdatetime = source.aestrdatetime AND target.hlgtn = source.hlgtn").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
#display the table
display(spark.sql(f"select * from {full_table_name} limit 10"))

subjid,aeout_n,aeseryn,aestrda,aestrdatetime,aestrtime,hlgtn,lltcur,center,comment,source_file,loaded_at,ingestion_id
840414002,1,0,2004-09-30,2004-09-30 07:37:51,07:37:51,null,Y,414,A big string (len 300) Lorem ipsum dolor sit amet; consectetur adipiscing elit. Integer nec odio. Praesent libero. Sed cursus ante dapibus diam. Sed nisi. Nulla quis sem at nibh elementum imperdiet. Duis sagittis ipsum. Praesent mauris. Fusce nec tellus sed augue semper porta. Mauris massa. Vestibul,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
826290005,3,0,null,null,null,GENERAL SYSTEM DISORDERS NEC,Y,290,null,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
826290002,1,0,null,null,null,null,Y,290,null,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
616187066,1,0,null,null,null,ANXIETY DISORDERS AND SYMPTOMS,Y,188,null,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
616187061,1,0,null,null,null,HEADACHES,Y,189,null,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
616187053,1,0,null,null,null,CARDIAC ARRHYTHMIAS,Y,188,null,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
616187050,1,0,null,null,null,INFECTIONS - PATHOGEN CLASS UNSPECIFIED,Y,188,null,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
616187049,1,1,null,null,null,CARDIAC ARRHYTHMIAS,Y,189,null,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
616187048,1,0,null,null,null,CARDIAC ARRHYTHMIAS,Y,187,null,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
616187043,1,0,null,null,null,CARDIAC ARRHYTHMIAS,Y,189,null,ae.csv,2026-01-04T13:51:42.108Z,20260104_135139
